# D4 — Flask API + Railway.app Deployment

### Architecture
- Flask API loads saved model, encoders, and thresholds from Drive
- Scores incoming loan applications in real-time
- Generates LangChain/Mistral alert for High risk applications
- LangSmith traces every LLM call
- Deployed to Railway.app — permanent stable URL

### Endpoints
- POST /score — returns default_probability, risk_tier, alert_text
- GET /health — confirms API is live

In [1]:
# Install dependencies
!pip install flask langchain langchain-mistralai langsmith pyngrok scikit-learn imbalanced-learn -q
from google.colab import drive


# Gdrive mount
drive.mount('/content/drive')
print("Dependencies ready")
print("LangSmith tracing enabled")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dependencies ready
LangSmith tracing enabled


In [2]:
import os
from google.colab import userdata
from langchain_mistralai import ChatMistralAI

# LangSmith observability setup and API Key Setup
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_PROJECT"] = "FinSight_LoanDecider"

# Mistral API Key setup
llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=userdata.get('MISTRAL_API_KEY'),
    temperature=0.35
)




In [3]:
# Loading the encoders, model and thershold values

import sys
import sklearn
import json



base_path = '/content/drive/MyDrive/FDE Projects/FinSight_LoanDecider/D2_Classifier/Models'

import joblib
model = joblib.load(f'{base_path}/calibrated_rf_model.pkl')
print(f"Loaded: {type(model)}")
encoders = joblib.load(f'{base_path}/label_encoders.pkl')
with open(f'{base_path}/risk_thresholds.json') as f:
    thresholds = json.load(f)

print(f"Encoders: {list(encoders.keys())}")
print(f"Thresholds: {thresholds}")

Loaded: <class 'sklearn.calibration.CalibratedClassifierCV'>
Encoders: ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
Thresholds: {'low_threshold': 0.3, 'high_threshold': 0.6}


In [4]:
from flask import Flask, request, jsonify
import threading
import pandas as pd
import numpy as np
from langchain_core.prompts import PromptTemplate

app = Flask(__name__)

# Prompt
alert_template = PromptTemplate(
    input_variables=["loan_id", "person_age", "person_income",
                     "person_emp_length", "loan_amnt", "loan_int_rate",
                     "loan_percent_income", "cb_person_cred_hist_length", "cb_person_default_on_file",
                     "person_home_ownership","loan_intent","loan_grade"],
    template="""
You are a senior credit analyst at FinSight Capital writing an internal note.

Write a 3-sentence alert for the branch manager:
- Sentence 1: State the risk tier and loan purpose, explain why this application is concerning
- Sentence 2: Identify the two most concerning factors with specific numbers
- Sentence 3: Recommend one specific action before approval

Applicant Profile:
- Loan ID: {loan_id}
- Age: {person_age} years old
- Annual Income: ${person_income} | Employment Experience: {person_emp_length} years
- Home Ownership: {person_home_ownership}
- Credit History Length: {cb_person_cred_hist_length} years
- Default Status: {cb_person_default_on_file}
- Loan Amount: ${loan_amnt} | Purpose: {loan_intent} | Interest Rate: {loan_int_rate}% | Percent Income: {loan_percent_income}%
- Loan Grade: {loan_grade}

Rules:
- Do NOT use the words: model, algorithm, AI
- Write in flowing prose, no bullet points, no bold headers
- Write exactly 3 sentences
- 100 words maximum
- Read like a human credit analyst note
- Write in plain text only — no asterisks, no bold, no italic, no headers
- Do not start with "Alert" or "Credit Alert" or any label
- First word must be a regular English word starting the first sentence directly
"""
)

alert_chain = alert_template | llm
print("Prompt template ready")

features = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
            'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
            'cb_person_default_on_file', 'person_home_ownership',
            'loan_intent', 'loan_grade']

categorical_cols = ['person_home_ownership', 'cb_person_default_on_file', 'loan_grade', 'loan_intent']

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'live', 'project': 'FinSight_LoanDecider'})

@app.route('/score', methods=['POST'])
def score():
    data = request.get_json(silent=True)

    try:
        if not data:
            return jsonify({'error': 'Invalid or missing JSON body'}), 400

        required_fields = [
            'loan_id',
            'person_age',
            'person_income',
            'person_emp_length',
            'person_home_ownership',
            'cb_person_cred_hist_length',
            'cb_person_default_on_file',
            'loan_amnt',
            'loan_intent',
            'loan_int_rate',
            'loan_percent_income',
            'loan_grade'
        ]

        missing_fields = [field for field in required_fields if field not in data]
        if missing_fields:
            return jsonify({
                'error': 'Validation failed',
                'details': [f"Missing required field: {field}" for field in missing_fields]
            }), 400

        errors = []

        if not (20 <= data.get('person_age', 0) <= 80):
            errors.append('person_age must be between 20 and 80')

        if not (8000 <= data.get('person_income', 0) <= 271268):
            errors.append('person_income must be between 8,000 and 271,268')

        if not (500 <= data.get('loan_amnt', 0) <= 35000):
            errors.append('loan_amnt must be between 500 and 35,000')

        if not (5 <= data.get('loan_int_rate', 0) <= 20):
            errors.append('loan_int_rate must be between 5 and 20')

        if data.get('person_home_ownership') not in ['RENT', 'OWN', 'MORTGAGE', 'OTHER']:
            errors.append('invalid person_home_ownership value')

        if data.get('loan_intent') not in ['PERSONAL', 'EDUCATION', 'MEDICAL', 'VENTURE', 'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']:
            errors.append('invalid loan_intent value')

        if data.get('loan_grade') not in ['A', 'B', 'C', 'D', 'E', 'F', 'G']:
            errors.append('invalid loan_grade value')
        if errors:
            return jsonify({'error': 'Validation failed', 'details': errors}), 400

        input_data = {
            "loan_id": data['loan_id'],
            "person_age": data['person_age'],
            "person_income": data['person_income'],
            "person_emp_length": data['person_emp_length'],
            "loan_amnt": data['loan_amnt'],
            "loan_int_rate": data['loan_int_rate'],
            "loan_percent_income": data['loan_percent_income'],
            "cb_person_cred_hist_length": data['cb_person_cred_hist_length'],
            "cb_person_default_on_file": data['cb_person_default_on_file'],
            "person_home_ownership": data['person_home_ownership'],
            "loan_intent": data['loan_intent'],
            "loan_grade": data['loan_grade']
          }

        input_df = pd.DataFrame([input_data])

        # 2. Re-order and filter the DataFrame to match D2's 'features' list exactly
        # This drops any extra keys (like person_gender) and ensures the sequence is correct
        input_df = input_df[features]

        for col in categorical_cols:
            input_df[col] = encoders[col].transform(input_df[col])

        prob = float(model.predict_proba(input_df[features])[0][1])

        if prob >= thresholds['high_threshold']:
            tier = 'High'
        elif prob >= thresholds['low_threshold']:
            tier = 'Medium'
        else:
            tier = 'Low'

        alert_text = ""
        if tier == 'High':
            response = alert_chain.invoke({
                "loan_id": data['loan_id'],
                "person_age": data['person_age'],
                "person_income": f"{int(data['person_income']):,}",
                "person_emp_length": data['person_emp_length'],
                "person_home_ownership": data['person_home_ownership'],
                "cb_person_cred_hist_length": data['cb_person_cred_hist_length'],
                "cb_person_default_on_file": data['cb_person_default_on_file'],
                "loan_amnt": f"{int(data['loan_amnt']):,}",
                "loan_intent": data['loan_intent'],
                "loan_int_rate": data['loan_int_rate'],
                "loan_percent_income": data['loan_percent_income'],
                "loan_grade": data['loan_grade']
            })
            alert_text = response.content

        return jsonify({
            'loan_id': data.get('loan_id'),
            'default_probability': round(prob, 3),
            'risk_tier': tier,
            'flagged': tier == 'High',
            'alert_text': alert_text
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500



Prompt template ready


In [5]:
import os
import threading
from pyngrok import ngrok

# 1. Authenticate ngrok (Crucial step)
# Get your token from https://ngrok.com
# ngrok API KEY set up
from pyngrok import ngrok
ngrok.set_auth_token(userdata.get('NGROK_API_KEY'))

# 2. Clean up any existing ngrok tunnels and old port 5000 processes
ngrok.kill()
os.system("fuser -k 5000/tcp")

# 3. Start Flask in a background thread
# Ensure your 'app' variable is already defined above this cell
thread = threading.Thread(target=app.run, kwargs={"port": 5000, "use_reloader": False, "debug": False})
thread.daemon = True
thread.start()

# 4. Connect ngrok safely with error handling
try:
    public_url = ngrok.connect(5000)
    print(f"\n🚀 API live at: {public_url}")
    print(f"Health check: {public_url}/health")
    print(f"Score check: {public_url}/score")
except Exception as e:
    print(f"❌ Ngrok failed to connect. Error: {e}")


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit



🚀 API live at: NgrokTunnel: "https://impolite-creamer-reprint.ngrok-free.dev" -> "http://localhost:5000"
Health check: NgrokTunnel: "https://impolite-creamer-reprint.ngrok-free.dev" -> "http://localhost:5000"/health
Score check: NgrokTunnel: "https://impolite-creamer-reprint.ngrok-free.dev" -> "http://localhost:5000"/score


In [6]:
import requests

test_payload = {
    "loan_id": 1001,
    "person_age": 34,
    "person_income": 85000,
    "person_emp_length": 6,
    "person_home_ownership": "RENT",
    "cb_person_cred_hist_length": 8,
    "cb_person_default_on_file": 0,
    "loan_amnt": 25000,
    "loan_intent": "DEBTCONSOLIDATION",
    "loan_int_rate": 13.45,
    "loan_percent_income": 0.29,
    "loan_grade": "B"
}

response = requests.post(
    "https://impolite-creamer-reprint.ngrok-free.dev/score",
    json=test_payload,
    headers={
        "ngrok-skip-browser-warning": "true",
        "Content-Type": "application/json"
    }
)

print("Status code:", response.status_code)
print("Raw text:", response.text)

try:
    print("JSON:", response.json())
except Exception:
    print("Response is not JSON")

INFO:werkzeug:127.0.0.1 - - [29/Jun/2026 17:04:02] "POST /score HTTP/1.1" 200 -


Status code: 200
Raw text: {"alert_text":"","default_probability":0.401,"flagged":false,"loan_id":1001,"risk_tier":"Medium"}

JSON: {'alert_text': '', 'default_probability': 0.401, 'flagged': False, 'loan_id': 1001, 'risk_tier': 'Medium'}


In [14]:
high_risk_payload = {
    "loan_id": 1002,
    "person_age": 22,
    "person_income": 150000,
    "person_emp_length": 0,
    "person_home_ownership": "RENT",
    "cb_person_cred_hist_length": 10,
    "cb_person_default_on_file": 1,
    "loan_amnt": 10000,
    "loan_intent": "PERSONAL",
    "loan_int_rate": 8.5,
    "loan_percent_income": 0.067,
    "loan_grade": "A"
}

response = requests.post(
    "https://impolite-creamer-reprint.ngrok-free.dev/score",
    json=high_risk_payload,
    headers={
        "ngrok-skip-browser-warning": "true",
        "Content-Type": "application/json"
    }
)

print("Status code:", response.status_code)
print("JSON:", response.json())

INFO:werkzeug:127.0.0.1 - - [29/Jun/2026 17:37:48] "POST /score HTTP/1.1" 200 -


Status code: 200
JSON: {'alert_text': '', 'default_probability': 0.06, 'flagged': False, 'loan_id': 1002, 'risk_tier': 'Low'}
